In [37]:
import pandas as pd
import glob
from pathlib import Path
import os
from datetime import datetime

# Configuration des chemins
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold"/ "election"

GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dossier Silver : {SILVER_DIR}")
print(f"Dossier Gold : {GOLD_DIR}")

Dossier Silver : /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver
Dossier Gold : /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/election


In [38]:
# --- Création de Dim_Nuance ---
path_silver_nuance = SILVER_DIR / "dictionnaire_des_nuances_politiques_silver.csv"
df_nuance_silver = pd.read_csv(path_silver_nuance, sep=";")

# On garde uniquement les clés et les libellés pour le modèle en étoile
dim_nuance = df_nuance_silver[['id_nuance', 'libelle_nuance']].copy()

print(f"Taille Dim_Nuance : {dim_nuance.shape}")
display(dim_nuance.head())

# Sauvegarder dans la couche Gold
dim_nuance.to_csv(GOLD_DIR / "dim_nuance.csv", index=False, sep=";")
print("✅ dim_nuance.csv sauvegardé dans le dossier Gold.")

Taille Dim_Nuance : (86, 2)


,id_nuance,libelle_nuance
0,EXG,Extrême gauche
1,COM,Communiste
2,PG,Parti de Gauche
3,SOC,Socialiste
4,RDG,Radical de Gauche


✅ dim_nuance.csv sauvegardé dans le dossier Gold.


In [39]:
# 1. Lister tous les fichiers silver des élections présidentielles (2012, 2017, 2022)
silver_files = sorted(SILVER_DIR.glob("*-pres-t*-commune-rhone-69-silver.csv"))

print(f"Nombre de fichiers trouvés : {len(silver_files)}")
for f in silver_files:
    print(f"  ✓ {f.name}")

# 2. Charger et concaténer les DataFrames
df_list = []
for file in silver_files:
    df = pd.read_csv(file, sep=";")
    df_list.append(df)
    print(f"  Loaded {file.name}: {df.shape}")

df_all = pd.concat(df_list, ignore_index=True)
print(f"\n✅ Taille du jeu de données consolidé : {df_all.shape}")

# Afficher les années et tours présents
print(f"\nAnnées et tours présents :")
print(df_all.groupby(['annee_election', 'tour']).size())

Nombre de fichiers trouvés : 6
  ✓ 2012-pres-t1-commune-rhone-69-silver.csv
  ✓ 2012-pres-t2-commune-rhone-69-silver.csv
  ✓ 2017-pres-t1-commune-rhone-69-silver.csv
  ✓ 2017-pres-t2-commune-rhone-69-silver.csv
  ✓ 2022-pres-t1-commune-rhone-69-silver.csv
  ✓ 2022-pres-t2-commune-rhone-69-silver.csv
  Loaded 2012-pres-t1-commune-rhone-69-silver.csv: (2930, 33)
  Loaded 2012-pres-t2-commune-rhone-69-silver.csv: (586, 33)
  Loaded 2017-pres-t1-commune-rhone-69-silver.csv: (3080, 33)
  Loaded 2017-pres-t2-commune-rhone-69-silver.csv: (560, 33)
  Loaded 2022-pres-t1-commune-rhone-69-silver.csv: (3204, 33)
  Loaded 2022-pres-t2-commune-rhone-69-silver.csv: (534, 33)

✅ Taille du jeu de données consolidé : (10894, 33)

Années et tours présents :
annee_election  tour
2012            1       2930
                2        586
2017            1       3080
                2        560
2022            1       3204
                2        534
dtype: int64


In [40]:
# ✅ VERIFICATION: Check schema standardization before processing
print("="*120)
print("📋 SCHEMA VERIFICATION: Silver Layer Files")
print("="*120 + "\n")

silver_files_check = sorted(SILVER_DIR.glob("*-pres-t*-commune-rhone-69-silver.csv"))
expected_cols = {
    'code_departement', 'libelle_departement', 'code_commune', 'libelle_commune',
    'annee_election', 'tour', 'inscrits', 'abstentions', 'votants', 'blancs', 'nuls',
    'blancs_et_nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins',
    'pct_blancs_vot', 'pct_nuls_ins', 'pct_nuls_vot', 'pct_blancs_et_nuls_ins',
    'pct_blancs_et_nuls_vot', 'pct_exprimes_ins', 'pct_exprimes_vot',
    'numero_panneau', 'sexe', 'nom', 'prenom', 'voix', 'pct_voix_ins',
    'pct_voix_exprimes', 'nuance', 'source_dataset', 'date_refresh'
}

all_valid = True
for file in silver_files_check:
    df_check = pd.read_csv(file, sep=";", nrows=1)
    actual_cols = set(df_check.columns)
    
    is_valid = expected_cols.issubset(actual_cols)
    status = "✅" if is_valid else "❌"
    
    print(f"{status} {file.name}: {len(df_check.columns)} columns")
    
    if not is_valid:
        missing = expected_cols - actual_cols
        print(f"   Missing: {missing}")
        all_valid = False

if all_valid:
    print("\n✅ All silver files have standardized schema!")
else:
    print("\n⚠️  Some files have schema issues - check above")

print("="*120 + "\n")

📋 SCHEMA VERIFICATION: Silver Layer Files

✅ 2012-pres-t1-commune-rhone-69-silver.csv: 33 columns
✅ 2012-pres-t2-commune-rhone-69-silver.csv: 33 columns
✅ 2017-pres-t1-commune-rhone-69-silver.csv: 33 columns
✅ 2017-pres-t2-commune-rhone-69-silver.csv: 33 columns
✅ 2022-pres-t1-commune-rhone-69-silver.csv: 33 columns
✅ 2022-pres-t2-commune-rhone-69-silver.csv: 33 columns

✅ All silver files have standardized schema!



In [41]:
# Extraction des combinaisons uniques d'année et de tour
dim_election = df_all[['annee_election', 'tour']].drop_duplicates().reset_index(drop=True)

# Création d'une clé primaire (id_election)
dim_election['id_election'] = dim_election['annee_election'].astype(str) + "_T" + dim_election['tour'].astype(str)

# Réorganisation des colonnes
dim_election = dim_election[['id_election', 'annee_election', 'tour']]

print(f"Taille Dim_Election : {dim_election.shape}")
display(dim_election.head())

Taille Dim_Election : (6, 3)


,id_election,annee_election,tour
0,2012_T1,2012,1
1,2012_T2,2012,2
2,2017_T1,2017,1
3,2017_T2,2017,2
4,2022_T1,2022,1


In [42]:
# Extraction des informations géographiques
cols_geo = ['code_departement', 'libelle_departement', 'code_commune', 'libelle_commune']
dim_commune = df_all[cols_geo].drop_duplicates().reset_index(drop=True)

# Création d'une clé primaire (id_commune) formatée avec le standard INSEE (ex: 69001)
dim_commune['id_commune'] = dim_commune['code_departement'].astype(str) + dim_commune['code_commune'].astype(str).str.zfill(3)

# Réorganisation
dim_commune = dim_commune[['id_commune', 'code_departement', 'libelle_departement', 'code_commune', 'libelle_commune']]

print(f"Taille Dim_Commune : {dim_commune.shape}")
display(dim_commune.head())


Taille Dim_Commune : (577, 5)


,id_commune,code_departement,libelle_departement,code_commune,libelle_commune
0,69001,69,RHONE,1,Affoux
1,69010,69,RHONE,10,L'Arbresle
2,69100,69,RHONE,100,Irigny
3,69101,69,RHONE,101,Jarnioux
4,69102,69,RHONE,102,Joux


In [43]:
# Extraction des informations candidats
cols_candidat = ['nom', 'prenom', 'sexe']
dim_candidat = df_all[cols_candidat].drop_duplicates().reset_index(drop=True)

# L'identité d'un candidat est unique (nom, prénom, sexe). 
# La nuance politique (le parti) sera désormais portée directement par la table de faits.
dim_candidat['id_candidat'] = dim_candidat.index + 1 # Clé primaire auto-incrémentée
dim_candidat['id_candidat'] = "CAND_" + dim_candidat['id_candidat'].astype(str).str.zfill(3)

# Réorganisation
dim_candidat = dim_candidat[['id_candidat', 'nom', 'prenom', 'sexe']]

print(f"Taille Dim_Candidat : {dim_candidat.shape}")
display(dim_candidat.head())


Taille Dim_Candidat : (20, 4)


,id_candidat,nom,prenom,sexe
0,CAND_001,JOLY,Eva,F
1,CAND_002,LE PEN,Marine,F
2,CAND_003,SARKOZY,Nicolas,M
3,CAND_004,MÉLENCHON,Jean-Luc,M
4,CAND_005,POUTOU,Philippe,M


In [44]:
# Jointures pour récupérer les clés primaires (Foreign Keys)
fact_df = df_all.copy()

# 1. Jointure avec Dim_Election
fact_df['id_election'] = fact_df['annee_election'].astype(str) + "_T" + fact_df['tour'].astype(str)

# 2. Jointure avec Dim_Commune (Format INSEE 5 caractères)
fact_df['id_commune'] = fact_df['code_departement'].astype(str) + fact_df['code_commune'].astype(str).str.zfill(3)

# 3. Jointure avec Dim_Candidat (uniquement sur le nom/prenom/sexe)
fact_df = fact_df.merge(
    dim_candidat, 
    on=['nom', 'prenom', 'sexe'], 
    how='left'
)

# 4. Renommage de la nuance
fact_df = fact_df.rename(columns={'nuance': 'id_nuance'})

# --- SPLIT DES TABLES DE FAITS (GRAINS DIFFERENTS) ---

# Table : Fact Participation (Grain = id_election, id_commune)
# Includes blancs_et_nuls column for harmonized historical data (2012 had aggregated blancs/nuls)
cols_participation = [
    'inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'blancs_et_nuls', 'exprimes',
    'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_blancs_vot',
    'pct_nuls_ins', 'pct_nuls_vot', 'pct_blancs_et_nuls_ins', 'pct_blancs_et_nuls_vot',
    'pct_exprimes_ins', 'pct_exprimes_vot'
]

fact_participation = fact_df[['id_commune', 'id_election'] + cols_participation].drop_duplicates().copy()
fact_participation['date_integration_gold'] = datetime.now().isoformat()

print(f"Taille Fact_Participation : {fact_participation.shape}")
print(f"Colonnes: {fact_participation.columns.tolist()}")
display(fact_participation.head(3))

# Table : Fact Resultats (Grain = id_election, id_commune, id_candidat)
cols_resultats = ['voix', 'pct_voix_ins', 'pct_voix_exprimes']

fact_resultats = fact_df[['id_commune', 'id_election', 'id_candidat', 'id_nuance'] + cols_resultats].copy()
fact_resultats['date_integration_gold'] = datetime.now().isoformat()

print(f"\nTaille Fact_Resultats : {fact_resultats.shape}")
display(fact_resultats.head(3))

Taille Fact_Participation : (1680, 20)
Colonnes: ['id_commune', 'id_election', 'inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'blancs_et_nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_blancs_vot', 'pct_nuls_ins', 'pct_nuls_vot', 'pct_blancs_et_nuls_ins', 'pct_blancs_et_nuls_vot', 'pct_exprimes_ins', 'pct_exprimes_vot', 'date_integration_gold']


,id_commune,id_election,inscrits,abstentions,votants,blancs,nuls,blancs_et_nuls,exprimes,pct_abs_ins,pct_vot_ins,pct_blancs_ins,pct_blancs_vot,pct_nuls_ins,pct_nuls_vot,pct_blancs_et_nuls_ins,pct_blancs_et_nuls_vot,pct_exprimes_ins,pct_exprimes_vot,date_integration_gold
0,69001,2012_T1,250,25,225,7,0,7,218,10.00,90.00,2.80,3.11,0.0,0.0,2.80,3.11,87.20,96.89,2026-03-28T02:12:43.523187
10,69010,2012_T1,3858,674,3184,65,0,65,3119,17.47,82.53,1.68,2.04,0.0,0.0,1.68,2.04,80.84,97.96,2026-03-28T02:12:43.523187
20,69100,2012_T1,5948,873,5075,81,0,81,4994,14.68,85.32,1.36,1.60,0.0,0.0,1.36,1.60,83.96,98.40,2026-03-28T02:12:43.523187



Taille Fact_Resultats : (10894, 8)


,id_commune,id_election,id_candidat,id_nuance,voix,pct_voix_ins,pct_voix_exprimes,date_integration_gold
0,69001,2012_T1,CAND_001,ECO,3,1.2,1.38,2026-03-28T02:12:43.538050
1,69001,2012_T1,CAND_002,FN,61,24.4,27.98,2026-03-28T02:12:43.538050
2,69001,2012_T1,CAND_003,UMP,68,27.2,31.19,2026-03-28T02:12:43.538050


In [45]:
# Sauvegarde des dimensions
dim_election.to_csv(GOLD_DIR / "dim_election.csv", index=False, sep=";")
dim_commune.to_csv(GOLD_DIR / "dim_commune.csv", index=False, sep=";")
dim_candidat.to_csv(GOLD_DIR / "dim_candidat.csv", index=False, sep=";")

# Sauvegarde des tables de faits scindées
fact_participation.to_csv(GOLD_DIR / "fact_participation.csv", index=False, sep=";")
fact_resultats.to_csv(GOLD_DIR / "fact_resultats_candidat.csv", index=False, sep=";")

print("\n" + "="*120)
print("✅ GOLD LAYER GENERATED SUCCESSFULLY")
print("="*120)
print(f"\n📊 DATA SUMMARY:")
print(f"  • Dim_Election: {dim_election.shape[0]} rows (2012 T1/T2, 2017 T1/T2, 2022 T1/T2)")
print(f"  • Dim_Commune: {dim_commune.shape[0]} unique communes")
print(f"  • Dim_Candidat: {dim_candidat.shape[0]} unique candidates")
print(f"  • Dim_Nuance: Loaded from silver layer dictionary")
print(f"\n📈 FACT TABLES:")
print(f"  • Fact_Participation: {fact_participation.shape[0]} rows (1 row per commune+election)")
print(f"  • Fact_Resultats_Candidat: {fact_resultats.shape[0]} rows (1 row per candidate+commune+election)")
print(f"\n📁 Location: {GOLD_DIR}/")
print("="*120 + "\n")


✅ GOLD LAYER GENERATED SUCCESSFULLY

📊 DATA SUMMARY:
  • Dim_Election: 6 rows (2012 T1/T2, 2017 T1/T2, 2022 T1/T2)
  • Dim_Commune: 577 unique communes
  • Dim_Candidat: 20 unique candidates
  • Dim_Nuance: Loaded from silver layer dictionary

📈 FACT TABLES:
  • Fact_Participation: 1680 rows (1 row per commune+election)
  • Fact_Resultats_Candidat: 10894 rows (1 row per candidate+commune+election)

📁 Location: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/election/

